In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
INPUT_ARTS = BASE_PATH + "processed_v2/articles_processed.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("1. Khoi tao Spark Session cho Categorical CBF...")
spark = SparkSession.builder \
    .appName("Retrieval_Categorical_Profile") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

Mounted at /content/drive
1. Khoi tao Spark Session cho Categorical CBF...


In [ ]:
print("2. Doc du lieu va chia khung thoi gian...")
transactions = spark.read.parquet(INPUT_TRANS)
articles = spark.read.parquet(INPUT_ARTS)

max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)

# Cua so 90 ngay de hoach dinh ro "Gu" an mac (Categorical Profile) cua khach
train_hist_start = val_start - datetime.timedelta(days=90)
test_hist_start = test_start - datetime.timedelta(days=90)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

print(f"Train History: {train_hist_start} -> {val_start}")
print(f"Test History:  {test_hist_start} -> {test_start}")

2. Doc du lieu va chia khung thoi gian...
Train History: 2020-06-10 -> 2020-09-08
Test History:  2020-06-17 -> 2020-09-15


In [ ]:
def generate_categorical_candidates(history_df, articles_df, top_n=40):
    # Buoc 1: Chon dac trung va tao Combo_ID lam min hon dua tren cac cot dang co
    arts_meta = articles_df.select(
        "article_id",
        "gender",                # Giới tính
        "product_group_name",    # Nhóm sản phẩm
        "product_type_name",     # Loại sản phẩm chi tiết
        "colour_group_name"      # Màu sắc
    )

    # Tao ma to hop (Combo ID) mo rong
    arts_meta = arts_meta.withColumn(
        "combo_id",
        F.concat_ws("_",
                    F.col("gender"),
                    F.col("product_group_name"),
                    F.col("product_type_name"),
                    F.col("colour_group_name"))
    )

    hist_arts = history_df.join(arts_meta, "article_id", "inner")

    # Lay moc thoi gian lon nhat trong tap lich su
    hist_max_date = history_df.select(F.max("t_dat_date")).collect()[0][0]


    # Buoc 2: Ho so khach hang voi trong so thoi gian (Time Decay)
    hist_arts = hist_arts.withColumn(
        "days_ago",
        F.datediff(F.lit(hist_max_date), F.col("t_dat_date"))
    ).withColumn(
        "decay_weight",
        F.pow(0.95, F.col("days_ago")) # Giao dich cang moi, diem cang cao
    )

    user_profile = hist_arts.groupBy("customer_id", "combo_id") \
                            .agg(F.sum("decay_weight").alias("user_affinity"))

    window_user_combo = Window.partitionBy("customer_id").orderBy(F.col("user_affinity").desc())

    # Giu lai Top 5 Combo yeu thich nhat
    user_top_combos = user_profile.withColumn("rn", F.row_number().over(window_user_combo)) \
        .filter(F.col("rn") <= 5).drop("rn")


    # Buoc 3: Do nong san pham voi gia toc ban hang (Trend Velocity)

    recent_14d_start = hist_max_date - datetime.timedelta(days=14)
    recent_7d_start = hist_max_date - datetime.timedelta(days=7)

    # Chi xet giao dich 14 ngay gan nhat
    recent_hist_arts = hist_arts.filter(F.col("t_dat_date") >= recent_14d_start)

    item_sales = recent_hist_arts.groupBy("combo_id", "article_id").agg(
        # Luot ban 7 ngay gan nhat (Tuan 2)
        F.sum(F.when(F.col("t_dat_date") >= recent_7d_start, 1).otherwise(0)).alias("sales_7d"),
        # Luot ban 7 ngay truoc do (Tuan 1)
        F.sum(F.when(F.col("t_dat_date") < recent_7d_start, 1).otherwise(0)).alias("sales_prev_7d")
    )

    # Tinh gia toc (Velocity) va Diem Hotness
    combo_popularity = item_sales.withColumn(
        "velocity",
        F.col("sales_7d") / (F.col("sales_prev_7d") + 1.0) # +1.0 de tranh chia cho 0
    ).withColumn(
        "item_hotness",
        F.col("sales_7d") * F.col("velocity")
    )

    window_combo_item = Window.partitionBy("combo_id").orderBy(F.col("item_hotness").desc())

    # Giu lai Top 20 item hot nhat trong moi Combo
    trending_items_per_combo = combo_popularity.withColumn("rn", F.row_number().over(window_combo_item)) \
        .filter(F.col("rn") <= 20).drop("rn")


    # Buoc 4 va 5: Ghep noi va Tinh diem Proxy moi
    candidates = user_top_combos.join(trending_items_per_combo, "combo_id", "inner")

    # Tinh diem tong hop co xet den thoi gian va gia toc
    candidates = candidates.withColumn(
        "lr_proxy_score",
        F.col("user_affinity") * F.col("item_hotness")
    )

    window_final = Window.partitionBy("customer_id").orderBy(F.col("lr_proxy_score").desc())

    # Cat lay Top N ung vien va tra ve
    final_cands = candidates.withColumn("rn", F.row_number().over(window_final)) \
        .filter(F.col("rn") <= top_n) \
        .select("customer_id", "article_id", "lr_proxy_score") \
        .withColumn("strategy", F.lit("optimized_categorical"))

    return final_cands

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"   -> Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
print("\n3. Generating Categorical candidates for Train set...")
train_cands = generate_categorical_candidates(train_hist_df, articles, top_n=40)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_categorical.parquet")

print("4. Generating Categorical candidates for Test set...")
test_cands = generate_categorical_candidates(test_hist_df, articles, top_n=40)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_categorical.parquet")

print("5. Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)
print("Hoan tat! Luong dac trung Categorical da duoc mo rong.")


3. Generating Categorical candidates for Train set...
4. Generating Categorical candidates for Test set...
5. Evaluating TEST set:
   -> Actuals: 207996 | Hits: 6273 | Recall: 0.0302
Hoan tat! Luong dac trung Categorical da duoc mo rong.
